In [ ]:
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt 
import subprocess
import matplotlib as mpl
import os
import re
import shlex
# import rcr_Suport as rcr
import copy
import shutil
import time
import shutil

In [ ]:
# ============================================================
# CONFIGURATION — All tuneable parameters in one place
# ============================================================
# Paper algorithms use the public config interface. All clinical values and paths
# come from git-ignored config/local.py; copy config/local_example.py for its schema.
from config import load_case

CASE = os.environ.get("AORTA_CASE")
if not CASE:
    raise RuntimeError("Set AORTA_CASE to a label defined in private config/local.py")
cfg = load_case(CASE)

patient_number = cfg["dataset_id"]

inp_path = os.path.join(cfg["sim_dir"], f"{patient_number}_CSM_calib")

interafce_mesh = pv.read(os.path.join(
    cfg["sim_dir"], f"{patient_number}_SD-mesh-complete", "mesh-surfaces", "S_interface.vtp"))

# ---- objective volume variation (target deformed mesh at systole) ----
path_def_objective = os.path.join(
    cfg["deformed_mesh_dir"], f"mw_{cfg['stiffness_target_phase_index']}.vtp"
)

# --- Initial guess for elastic modulus ----------------------
E_INITIAL = 2.5e7                   # dyn/cm^2
E_mean = E_INITIAL
# --- Finite-difference Jacobian -----------------------------
PERTURBATION_FRACTION = 0.05        # delta = 5 % of current E_i
# --- Newton update ------------------------------------------
LEARNING_RATE  = 1.0                # damping factor on Newton step

# --- Convergence -------------------------------------------
MAX_ITERATIONS    = 30              # outer Newton iterations
CONVERGENCE_TOL   = 0.01            # stop when max |relative error| < 1 %
CONSECUTIVE_CONV  = 3               # must satisfy tolerance this many times in a row

# --- Parameter bounds --------------------------------------
E_MIN = 1e4
E_MAX = 1e9
# --- Simulation --------------------------------------------
N_PROCS = cfg["n_procs"]            # MPI processes for svFSI


In [ ]:
def read_cuts_posit(file_path):
    # Check if the file exists
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"The file '{file_path}' does not exist.")
    
    data = []
    with open(file_path, "r") as f:
        # Read the header
        header = f.readline().strip().split("\t")
        # Read the data rows
        for line in f:
            row = line.strip().split("	")
            first=row[1][1:-1].split(' ')
            second=row[2][1:-1].split(' ')
            
            def remotion(array):
                arrayR=[]
                for i in array:
                    if not i=='':
                        arrayR.append(float(i))
                return arrayR
            
            first=remotion(first)
            second=remotion(second) 

            data.append([first,second])

    return data



file_path = cfg["cuts_posit"]

data = read_cuts_posit(file_path)
plane_info_arc=np.array(data[1])
print(plane_info_arc)


def calculate_volume(cut_posit,interface,inp_path,iteration=None):
    if iteration is None:
        result_file=inp_path
    else:
        result_file = os.path.join(
            inp_path, f"iteration_{iteration}", "csm", f"{N_PROCS}-procs",
            f"CSM_{cfg['csm_result_step']}.vtu",
        )
    if not os.path.isfile(result_file):
        raise FileNotFoundError(f"expected structural result was not created: {result_file}")
    mesh = pv.read(result_file)

    #! check ivnersion
    interface_T=interface.clip(normal=cut_posit[1], origin=cut_posit[0], inplace=False,invert=False)
    
    def _fill_mesh_holes(dataset, hole_size=1e9):
        poly = dataset.extract_surface().triangulate().clean()
        return poly.fill_holes(hole_size=hole_size).clean()

    interface_T = _fill_mesh_holes(interface_T)
    
    
    sampled_interface = interface_T.sample(mesh)

    interface_T_def = sampled_interface.warp_by_vector("Displacement", factor=1.0, inplace=False)

    # plotter = pv.Plotter(notebook=False)
    # # plotter.add_mesh(mesh_T, color='lightblue', opacity=0.7, label='Clipped Mesh')
    # plotter.add_mesh(interface_T, color='red', opacity=0.7, label='Interface')
    # plotter.add_mesh(interface_T_def, color='blue', opacity=0.7, label='Deformed Interface')
    # plotter.add_mesh(mesh, color='lightblue', opacity=0.2, label='Clipped Mesh')
    # plotter.add_legend()
    # plotter.show()
    
    volume_init = interface_T.volume
    volume_syst = interface_T_def.volume

    return volume_syst-volume_init

TARGET_VOLUME = calculate_volume(plane_info_arc, interafce_mesh, path_def_objective)
TARGET_VOLUME_SCALE = abs(TARGET_VOLUME)
if TARGET_VOLUME_SCALE <= np.finfo(float).eps:
    raise ValueError("target volume change must be non-zero")
print(TARGET_VOLUME)

In [ ]:
def write_in_file(path, elastic_modulus): 
    prestress_inp = f"{path}\\CSM_prestress\\prest_CSM.inp"

    
    with open(prestress_inp, 'r') as file1:
        content = file1.read()

    pattern = r'   Elasticity modulus:.*$'
    replacement = f"   Elasticity modulus: {elastic_modulus}"
    content = re.sub(pattern, replacement, content, flags=re.MULTILINE)

    with open(prestress_inp , 'w') as file2:
        file2.write(content)
    # ----------------------------------------------
    csm_inp = f"{path}\\CSM_iteract\\CSM.inp"
    with open(csm_inp, 'r') as file1:
        content = file1.read()

    pattern = r'   Elasticity modulus:.*$'
    replacement = f"   Elasticity modulus: {elastic_modulus}"
    content = re.sub(pattern, replacement, content, flags=re.MULTILINE)
    with open(csm_inp , 'w') as file2:
        file2.write(content)
        
    print("Elastic modulus written to input files: ", elastic_modulus)

def clear_folders_old(path):
    """
    Removes all folders inside 'path' that start with 'iteration'.
    """
    if not os.path.isdir(path):
        raise ValueError(f"Invalid directory: {path}")

    for name in os.listdir(path):
        full_path = os.path.join(path, name)

        if os.path.isdir(full_path) and name.startswith("iteration"):
            shutil.rmtree(full_path)

def run_simulation_ubuntu(inp_path, iteration):

    # The solver binary and the rank count come from config/local.py, like every
    # other path in this repository; both were hardcoded here, and the rank count
    # silently disagreed with N_PROCS a few cells above.
    svfsi = cfg["paths"]["svfsi_bin"]

    # Convert Windows path to WSL path so bash commands work reliably.
    inp_path_wsl = subprocess.check_output(
        ["wsl", "wslpath", "-a", inp_path],
        text=True,
    ).strip()

    work_dir = f"{inp_path_wsl}/iteration_{iteration}"
    prestress_inp = f"{inp_path_wsl}/CSM_prestress/prest_CSM.inp"
    # svFSI names the output folder after the rank count, so this has to follow
    # N_PROCS as well; it was hardcoded to 10-procs.
    prestress_step = cfg["prestress_result_step"]
    prestress_result = (
        f"{work_dir}/prestress/{N_PROCS}-procs/result_{prestress_step}.vtu"
    )
    csm_result = f"{inp_path_wsl}/result_{prestress_step}.vtu"
    csm_inp = f"{inp_path_wsl}/CSM_iteract/CSM.inp"


    simulation_command = (
        "export OPENBLAS_NUM_THREADS=1 && "
        "export OMP_NUM_THREADS=2 && "
        f"mkdir -p {shlex.quote(work_dir + '/prestress')} && "
        f"mkdir -p {shlex.quote(work_dir + '/csm')} && "
        f"cd {shlex.quote(work_dir + '/prestress')} && "
        f"mpiexec -np {N_PROCS} {shlex.quote(svfsi)} {shlex.quote(prestress_inp)} && "
        f"test -f {shlex.quote(prestress_result)} && "
        f"cp {shlex.quote(prestress_result)} {shlex.quote(csm_result)} && "
        f"cd {shlex.quote(work_dir + '/csm')} && "
        f"mpiexec -np {N_PROCS} {shlex.quote(svfsi)} {shlex.quote(csm_inp)}"
    )

    subprocess.run(["wsl", "bash", "-lc", simulation_command], check=True)
    return 0

In [ ]:
# def select_cut_ataa():

#     centerline_points = centerline_mesh.points
#     if centerline_points.shape[0] < 3:
#         raise ValueError("Centerline must contain at least 3 points to define a clipping plane.")

#     centerline_mesh["index"] = np.arange(centerline_points.shape[0])  # Add index for selection

#     plotter = pv.Plotter(notebook=False)

#     center = centerline_points.mean(axis=0)
#     vector1 = center - centerline_points[0]
#     vector2 = centerline_points[-2] - centerline_points[0]
#     normalc = np.cross(vector1, vector2)
#     normal_norm = np.linalg.norm(normalc)
#     if normal_norm == 0:
#         raise ValueError("Computed normal has zero magnitude; check centerline geometry.")
#     normalc = normalc / normal_norm

#     plotter.add_mesh(mesh.clip(origin=center, normal=normalc), opacity=0.5, color="lightgrey")
#     plotter.add_mesh(centerline_mesh, color="red", point_size=5)
#     plotter.show()

# select_cut_ataa()

In [ ]:
E_current = float(E_mean)

E_history = []
error_history = []
norm_history = []

converged_count = 0
#! cleaer folders 

for iteration in range(MAX_ITERATIONS):
    print(f"\n{'-'*60}")
    print(f"ITERATION {iteration + 1}/{MAX_ITERATIONS}")

    # Central finite-difference perturbations around current parameter.
    delta_E_fd = PERTURBATION_FRACTION * max(abs(E_current), 1.0)
    E_minus = float(np.clip(E_current - delta_E_fd, E_MIN, E_MAX))
    E_plus = float(np.clip(E_current + delta_E_fd, E_MIN, E_MAX))

    print("  Simulation 1/3: baseline (E_mean) ...")
    write_in_file(inp_path, E_current)
    run_simulation_ubuntu(inp_path, iteration=(str(iteration) + "_mean"))
    mean_volume = calculate_volume(
        plane_info_arc, interafce_mesh, inp_path, iteration=(str(iteration) + "_mean")
    )

    print("  Simulation 2/3: baseline (E_min) ...")
    write_in_file(inp_path, E_minus)
    run_simulation_ubuntu(inp_path, iteration=(str(iteration) + "_min"))
    min_volume = calculate_volume(
        plane_info_arc, interafce_mesh, inp_path, iteration=(str(iteration) + "_min")
    )

    print("  Simulation 3/3: baseline (E_max) ...")
    write_in_file(inp_path, E_plus)
    run_simulation_ubuntu(inp_path, iteration=(str(iteration) + "_max"))
    max_volume = calculate_volume(
        plane_info_arc, interafce_mesh, inp_path, iteration=(str(iteration) + "_max")
    )

    # Relative residual: r(E) = (target - model_volume(E)) / |target|
    error_mean = float((TARGET_VOLUME - mean_volume) / TARGET_VOLUME_SCALE)
    error_min = float((TARGET_VOLUME - min_volume) / TARGET_VOLUME_SCALE)
    error_max = float((TARGET_VOLUME - max_volume) / TARGET_VOLUME_SCALE)

    max_err = abs(error_mean)

    E_history.append([E_current])
    error_history.append([error_mean])
    norm_history.append(max_err)

    print(
        f"  Volumes: mean={mean_volume:.6f}, min={min_volume:.6f}, max={max_volume:.6f}"
    )
    print(
        f"  Errors : mean={error_mean:.6f}, min={error_min:.6f}, max={error_max:.6f}"
    )
    print(f"  Max |error| = {max_err:.6f}")

    if max_err < CONVERGENCE_TOL:
        converged_count += 1
        print(f"  Below tolerance ({converged_count}/{CONSECUTIVE_CONV})")
        if converged_count >= CONSECUTIVE_CONV:
            print(f"\nConverged after {iteration + 1} iterations.")
            break
    else:
        converged_count = 0

    # Central finite-difference Jacobian: J = dr/dE
    denom = E_plus - E_minus
    if abs(denom) < 1e-12:
        raise RuntimeError("Finite-difference denominator is too small. Check E bounds/perturbation.")
    J = (error_max - error_min) / denom

    safe_J = J if abs(J) > 1e-12 else np.sign(J + 1e-30) * 1e-12
    delta_E = -error_mean / safe_J

    E_new = float(np.clip(E_current + LEARNING_RATE * delta_E, E_MIN, E_MAX))

    print(f"  J = {J:.6e}")
    print(f"  Delta E = {LEARNING_RATE * delta_E:.6e}")
    print(f"  E_new   = {E_new:.6e}")

    E_current = E_new

else:
    print(f"\nDid not converge within {MAX_ITERATIONS} iterations.")
    if norm_history:
        print(f"Final max |error| = {norm_history[-1]:.6f}")

E_mean = E_current
E_min = float(np.clip(E_current * (1 - PERTURBATION_FRACTION), E_MIN, E_MAX))
E_max = float(np.clip(E_current * (1 + PERTURBATION_FRACTION), E_MIN, E_MAX))

print(f"\nFinal E = {E_current:.6e}")

In [ ]:
# ============================================================
# CONVERGENCE & PARAMETER HISTORY PLOTS
# ============================================================
N_PARAMS=1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Max |error| vs iteration --------------------------
ax = axes[0]
ax.semilogy(range(1, len(norm_history) + 1), np.array(norm_history) * 100, "o-k")
ax.axhline(CONVERGENCE_TOL * 100, color="r", ls="--", label=f"tol = {CONVERGENCE_TOL*100:.1f} %")
ax.set_xlabel("Iteration")
ax.set_ylabel("Max |relative error| (%)")
ax.set_title("Convergence history")
ax.legend()
ax.grid(True, which="both", ls=":")

# --- 2. Per-domain error vs iteration ---------------------
ax = axes[1]
err_arr = np.array(error_history) * 100  # (n_iter, N_PARAMS)
for d in range(N_PARAMS):
    ax.plot(range(1, len(error_history) + 1), err_arr[:, d], "o-", label=f"Domain {d+1}")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Iteration")
ax.set_ylabel("Relative error (%)")
ax.set_title("Per-domain error")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, ls=":")

# --- 3. Elastic modulus evolution -------------------------
ax = axes[2]
E_arr = np.array(E_history)  # (n_iter, N_PARAMS)
for d in range(N_PARAMS):
    ax.semilogy(range(1, len(E_history) + 1), E_arr[:, d], "o-", label=f"Domain {d+1}")
ax.set_xlabel("Iteration")
ax.set_ylabel("Elastic modulus E")
ax.set_title("Parameter evolution")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, which="both", ls=":")

plt.tight_layout()
plt.show()